# Semi-global primer trimming prototype

This notebook prototypes sequence-aware trimming for reads that AmpliGone could not trim at one or both ends using reference coordinates. It aligns the primer FASTA against 50 nt terminal windows from the adapter-cleaned `EQA_04.noadapters.fastq` with Parasail's affine-gap semi-global alignment.

The input FASTQ is never modified. Empty reads retained by adapter removal are preserved unchanged because Parasail cannot align an empty terminal window. The final cells write a separately trimmed FASTQ and a CSV containing each accepted primer call, score, identity, CIGAR, and cut length.

In [ ]:
import re

from collections.abc import Iterable, Iterator, Sequence
from pathlib import Path
from dataclasses import dataclass

import os
import pandas as pd
from Bio import SeqIO
from Bio.SeqIO.QualityIO import FastqGeneralIterator
from Bio.Seq import Seq

import parasail

from itertools import islice

In [ ]:
DIRECTORY_ROOT = Path(os.getcwd())
INPUT_FASTQ = DIRECTORY_ROOT / "manual_test/processed/EQA_04.subsampled.fastq"
PRIMER_FASTA = DIRECTORY_ROOT / "manual_test/ESIB_EQA_2026_SARS1.primers.fasta"
OUTPUT_FASTQ = DIRECTORY_ROOT / "manual_test/processed/EQA_04.subsampled.noprimers.fastq"
STATISTICS_CSV = DIRECTORY_ROOT / "manual_test/processed/EQA_04.sequence-aware.csv"

### Semi-global alignment settings

In [ ]:
MATCH_SCORE = 3
MISMATCH_SCORE = -2
GAP_OPEN = 3
GAP_EXTEND = 1


WINDOW_SIZE = 200
MIN_OVERLAP = 8
MIN_IDENTITY = 0.70
MIN_SCORE_FRACTION = 0.65
MAX_EXTRA_REFERENCE_BASES = 200

### Base functions and fastq dataclasses

In [ ]:
@dataclass(frozen=True)
class FastqRecord:
    title: str
    sequence: str
    qualities: str


@dataclass(frozen=True)
class TerminalAlignment:
    primer_name: str
    primer_sequence: str
    score: int
    overlap_bases: int
    matches: int
    identity: float
    cut: int
    cigar: str


@dataclass(frozen=True)
class TrimDecision:
    record: FastqRecord
    left_hit: TerminalAlignment | None
    right_hit: TerminalAlignment | None
    left_removed: int
    right_removed: int


def read_primers(path: Path) -> list[tuple[str, str]]:
    primers = [(record.id, str(record.seq).upper()) for record in SeqIO.parse(path, "fasta")]
    if not primers:
        raise ValueError(f"No primer records found in {path}")
    return primers


def read_fastq(path: Path) -> Iterator[FastqRecord]:
    with path.open() as handle:
        for title, sequence, qualities in FastqGeneralIterator(handle):
            yield FastqRecord(title, sequence.upper(), qualities)


def score_fraction(hit: TerminalAlignment) -> float:
    maximum = MATCH_SCORE * hit.overlap_bases
    return hit.score / maximum if maximum else 0.0

In [ ]:
primers = read_primers(PRIMER_FASTA)

### Parasail scoring matrix

In [ ]:
SCORE_MATRIX = parasail.matrix_create("ACGTN", MATCH_SCORE, MISMATCH_SCORE)

### Alignment and decision code

`sg_qb_de` makes a primer-prefix overhang and the unused window suffix free. This anchors the aligned primer fragment at the read boundary while recovering primers truncated before sequencing began. The 3' end uses the same operation after reverse-complementing its terminal window.

The CIGAR parser excludes only those two free boundary operations. Internal mismatches and affine gaps count against identity. The assertions above verify exact, left-truncated, and reverse-end coordinate handling.

In [ ]:
def reverse_complement(sequence: str) -> str:
    # Reverse the sequence and replace each base with its complement.
    return str(Seq(sequence).reverse_complement())


def align_terminal_prefix(
    window: str, primer_name: str, primer: str
) -> TerminalAlignment:
    """Align primer to the left edge of a window with free primer-prefix overhang."""

    # Perform semi-global alignment with a free prefix on the primer.
    result = parasail.sg_trace_scan_sat(
        primer, window, GAP_OPEN, GAP_EXTEND, SCORE_MATRIX
    )

    # Decode the alignment CIGAR string returned by parasail.
    cigar = result.cigar.decode
    if isinstance(cigar, bytes):
        cigar = cigar.decode("ascii")

    # Convert CIGAR operations into (length, operation) pairs.
    operations = [
        (int(length), operation)
        for length, operation in re.findall(r"(\d+)([=XIDM])", cigar)
    ]

    # Remove unaligned primer-prefix insertions and terminal deletions.
    if operations and operations[0][1] == "I":
        operations = operations[1:]
    if operations and operations[-1][1] == "D":
        operations = operations[:-1]

    # Count matching bases and aligned query/reference bases.
    matches = sum(
        length for length, operation in operations if operation in {"=", "M"}
    )
    query_bases = sum(
        length
        for length, operation in operations
        if operation in {"=", "X", "M", "I"}
    )
    reference_bases = sum(
        length
        for length, operation in operations
        if operation in {"=", "X", "M", "D"}
    )

    # Calculate alignment size and the effective overlap.
    alignment_columns = sum(length for length, _ in operations)
    overlap_bases = min(query_bases, reference_bases)

    return TerminalAlignment(
        primer_name=primer_name,
        primer_sequence=primer,
        score=result.score,
        overlap_bases=overlap_bases,
        matches=matches,
        identity=matches / alignment_columns if alignment_columns else 0.0,
        cut=result.end_ref + 1,
        cigar=cigar,
    )


def best_terminal_alignment(record_name: str,
    window: str, primers: Sequence[tuple[str, str]]
) -> TerminalAlignment | None:
    # No alignment is possible for an empty window.
    if not window:
        return None

    accepted = []

    for primer_name, primer_sequence in primers:
        # Align the current primer to the terminal window.
        hit = align_terminal_prefix(window, primer_name, primer_sequence)

        # if "0a95139b-a66b-4e23-b007-5246e56b070e" in record_name:
        #     print(record_name, primer_name)
        #     print("hit_cut:", hit.cut)
        #     print("max_allowed_cut:", len(window) - len(primer_sequence))
        #     print("cut_within_allowed_range:", hit.cut <= (len(window) - len(primer_sequence)))
        # Keep only alignments that satisfy all quality thresholds.
        if (
            hit.overlap_bases >= MIN_OVERLAP
            and hit.identity >= MIN_IDENTITY
            and score_fraction(hit) >= MIN_SCORE_FRACTION
            and hit.cut <= len(window)
        ):
            accepted.append(hit)

    # Select the highest-scoring accepted alignment.
    return max(
        accepted,
        key=lambda hit: (hit.score, hit.overlap_bases, hit.identity),
        default=None,
    )


def trim_record(
    record: FastqRecord, primers: Sequence[tuple[str, str]]
) -> TrimDecision:
    # Search for a primer at the beginning of the read.
    left_window = record.sequence[:WINDOW_SIZE]

    # Reverse-complement the read end so it can be aligned like a left terminal.
    right_window_as_prefix = reverse_complement(record.sequence[-WINDOW_SIZE:])

    left_hit = best_terminal_alignment(record.title, left_window, primers)
    right_hit = best_terminal_alignment(record.title, right_window_as_prefix, primers)

    # Convert alignment endpoints into bases to remove.
    left_removed = left_hit.cut if left_hit else 0
    right_removed = right_hit.cut if right_hit else 0

    # Calculate the right boundary of the retained sequence.
    right_boundary = len(record.sequence) - right_removed

    # Prevent overlapping trims from removing the entire read unexpectedly.
    if left_removed > right_boundary:
        left_hit = None
        right_hit = None
        left_removed = 0
        right_removed = 0
        right_boundary = len(record.sequence)

    # Preserve sequence and quality strings using the same slice.
    trimmed_record = FastqRecord(
        title=record.title,
        sequence=record.sequence[left_removed:right_boundary],
        qualities=record.qualities[left_removed:right_boundary],
    )

    return TrimDecision(
        record=trimmed_record,
        left_hit=left_hit,
        right_hit=right_hit,
        left_removed=left_removed,
        right_removed=right_removed,
    )


def decision_row(original: FastqRecord, decision: TrimDecision) -> dict[str, object]:
    # Store trimming results and alignment diagnostics in tabular form.
    return {
        "read_name": original.title.split()[0],
        "original_length": len(original.sequence),
        "trimmed_length": len(decision.record.sequence),
        "left_removed": decision.left_removed,
        "right_removed": decision.right_removed,
        "left_primer": decision.left_hit.primer_name if decision.left_hit else None,
        "right_primer": decision.right_hit.primer_name if decision.right_hit else None,
        "left_score": decision.left_hit.score if decision.left_hit else None,
        "right_score": decision.right_hit.score if decision.right_hit else None,
        "left_identity": decision.left_hit.identity if decision.left_hit else None,
        "right_identity": decision.right_hit.identity if decision.right_hit else None,
        "left_cigar": decision.left_hit.cigar if decision.left_hit else None,
        "right_cigar": decision.right_hit.cigar if decision.right_hit else None,
    }


def process_records(
    records: Iterable[FastqRecord], primers: Sequence[tuple[str, str]]
) -> tuple[list[FastqRecord], pd.DataFrame]:
    trimmed_records = []
    diagnostics = []

    for original in records:
        # Trim each record and collect its diagnostics.
        decision = trim_record(original, primers)
        trimmed_records.append(decision.record)
        diagnostics.append(decision_row(original, decision))

    # Return trimmed records and diagnostics as a DataFrame.
    return trimmed_records, pd.DataFrame(diagnostics)


def write_fastq(records: Iterable[FastqRecord], path: Path) -> None:
    # Write records in standard four-line FASTQ format.
    with path.open("w") as handle:
        for record in records:
            handle.write(
                f"@{record.title}\n{record.sequence}\n+\n{record.qualities}\n"
            )

## Subsample test for debugging

A hit must align at least 15 primer/read bases, reach 80% gap-aware identity, and retain at least 65% of the perfect-match score. Its reference span may exceed primer length by no more than five bases, allowing a small terminal adapter or insertion without bridging a long unrelated region.

The following 250-read sample is intentionally kept as a quick calibration surface. Inspect the accepted CIGARs before changing scoring or thresholds.

In [ ]:
sample_records = list(islice(read_fastq(INPUT_FASTQ), 250))

sample_trimmed, sample_diagnostics = process_records(sample_records, primers)


sample_summary = pd.Series(
    {
        "primers": len(primers),
        "sample_reads": len(sample_records),
        "reads_trimmed": int(
            ((sample_diagnostics.left_removed + sample_diagnostics.right_removed) > 0).sum()
        ),
        "left_ends_trimmed": int((sample_diagnostics.left_removed > 0).sum()),
        "right_ends_trimmed": int((sample_diagnostics.right_removed > 0).sum()),
        "both_ends_trimmed": int(
            (
                (sample_diagnostics.left_removed > 0)
                & (sample_diagnostics.right_removed > 0)
            ).sum()
        ),
    }
)


print(sample_summary.to_string())

display(
    sample_diagnostics.loc[
        (sample_diagnostics.left_removed > 0)
        | (sample_diagnostics.right_removed > 0)
    ].head(10)
)

## Process the complete FASTQ

This pass processes all adapter-cleaned input reads, preserves names and synchronized quality strings, and writes:

- `manual_test/processed/EQA_04.sequence-aware.fastq`
- `manual_test/processed/EQA_04.sequence-aware.csv`

The CSV is the audit trail for threshold tuning; rejected candidates and empty reads do not modify a read.

In [ ]:
# Read all input FASTQ records into memory.
all_records = list(read_fastq(INPUT_FASTQ))

# Trim detected primers and collect per-read diagnostics.
trimmed_records, statistics = process_records(all_records, primers)

# Write the trimmed reads to the output FASTQ file.
write_fastq(trimmed_records, OUTPUT_FASTQ)

# Save alignment and trimming diagnostics as CSV.
statistics.to_csv(STATISTICS_CSV, index=False)

# Identify reads where at least one end was trimmed.
trimmed_mask = (statistics.left_removed + statistics.right_removed) > 0

# Summarize trimming results for the complete input.
run_summary = pd.Series(
    {
        "input_reads": len(all_records),
        "reads_trimmed": int(trimmed_mask.sum()),
        "reads_unchanged": int((~trimmed_mask).sum()),
        "left_ends_trimmed": int((statistics.left_removed > 0).sum()),
        "right_ends_trimmed": int((statistics.right_removed > 0).sum()),
        "both_ends_trimmed": int(
            (
                (statistics.left_removed > 0)
                & (statistics.right_removed > 0)
            ).sum()
        ),
        "bases_removed": int(
            (statistics.left_removed + statistics.right_removed).sum()
        ),
    }
)

# Display the summary and output locations.
print(run_summary.to_string())
print(f"\nFASTQ: {OUTPUT_FASTQ.relative_to(DIRECTORY_ROOT)}")
print(f"Diagnostics: {STATISTICS_CSV.relative_to(DIRECTORY_ROOT)}")

## Output validation

Read the generated FASTQ back from disk and reconcile it with both the source records and the diagnostics table.

In [ ]:
written_records = list(read_fastq(OUTPUT_FASTQ))

observed_removed_bases = sum(
    len(original.sequence) - len(written.sequence)
    for original, written in zip(all_records, written_records)
)

assert observed_removed_bases == run_summary["bases_removed"]
assert OUTPUT_FASTQ.stat().st_size > 0
assert STATISTICS_CSV.stat().st_size > 0

print(
    f"Validated {len(written_records):,} FASTQ records and "
    f"{observed_removed_bases:,} removed bases."
)